# Evaluacion de Tamano de Chunk - RAG Histologia
**Grupo 2 | CETEC UBATIC**

Evalua 3 estrategias de chunking usando RAGAS + Groq como LLM evaluador.

## Flujo
1. Instalar dependencias
2. Cargar credenciales
3. Cargar CLIP (mismo modelo que el sistema principal)
4. Indexar el PDF en 3 colecciones Qdrant
5. Construir golden set desde el PDF real
6. Evaluar con RAGAS y comparar resultados

> **Prerequisito:** subir el PDF del manual `histologia_completo` a `/content/pdf/` antes de correr.

## Celda 1 — Instalar dependencias
> Correr solo una vez por sesion. Tarda ~2 minutos.

In [30]:
!pip install --quiet \
    langchain langchain-groq langchain-qdrant \
    ragas datasets sentence-transformers \
    transformers torch torchvision \
    pdfplumber pdf2image Pillow qdrant-client

!apt-get install -y -q poppler-utils

print('Dependencias instaladas')

Reading package lists...
Building dependency tree...
Reading state information...
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
Dependencias instaladas


## Celda 2 — Credenciales
> Requiere los siguientes secretos en Colab:
> `GROQ_API_KEY`, `QDRANT_URL`, `QDRANT_KEY`, `LANGCHAIN_API_KEY` (opcional)

In [31]:
import os
import glob
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

QDRANT_URL = userdata.get('QDRANT_URL')
QDRANT_KEY = userdata.get('QDRANT_KEY')

langsmith_key = userdata.get('LANGCHAIN_API_KEY')
if langsmith_key:
    os.environ['LANGCHAIN_TRACING_V2']  = 'true'
    os.environ['LANGCHAIN_ENDPOINT']    = 'https://api.smith.langchain.com'
    os.environ['LANGCHAIN_API_KEY']     = langsmith_key
    os.environ['LANGCHAIN_PROJECT']     = 'Evaluacion_ChunkSize_Histologia'
    print('LangSmith activado')
else:
    print('LangSmith no configurado - tracing desactivado')

pdfs = glob.glob('/content/pdf/*.pdf')
if pdfs:
    PDF_PATH = pdfs[0]
    print(f'PDF encontrado: {PDF_PATH}')
else:
    raise FileNotFoundError('No hay PDFs en /content/pdf/ - subi el manual primero')

LangSmith activado
PDF encontrado: /content/pdf/histologia_completo.pdf


## Celda 3 — Cargar CLIP
> Mismo modelo que usa el sistema principal (CETEC_g2).
> Necesario para generar embeddings de texto e imagen.

In [32]:
import torch
import numpy as np
from transformers import CLIPModel, CLIPTokenizer, CLIPProcessor
from PIL import Image

CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')

clip_model     = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(device)
clip_tokenizer = CLIPTokenizer.from_pretrained(CLIP_MODEL_NAME)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model.eval()

def embed_texto(texto: str) -> list:
    inputs = clip_tokenizer(
        [texto], padding=True, truncation=True,
        max_length=77, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        feats = clip_model.get_text_features(**inputs)
        if hasattr(feats, 'pooler_output'):
            emb = feats.pooler_output.cpu().numpy().flatten()
        else:
            emb = feats.cpu().numpy().flatten()
    return (emb / np.linalg.norm(emb)).tolist()

print('CLIP cargado')

Dispositivo: cpu


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP cargado


## Celda 4 - Inspeccionar el PDF
Corre esta celda para ver el contenido de cada pagina.
Usala para verificar que los numeros de pagina del golden set sean correctos.

In [33]:
import pdfplumber

print(f'Inspeccionando: {PDF_PATH}')
with pdfplumber.open(PDF_PATH) as pdf:
    print(f'Total de paginas: {len(pdf.pages)}')
    for i, page in enumerate(pdf.pages, 1):
        texto = page.extract_text()
        tablas = page.extract_tables()
        print(f'--- Pagina {i} ---')
        if texto:
            print(texto[:400])
        if tablas:
            print(f'  [Tablas: {len(tablas)}]')
            for t in tablas:
                for fila in t:
                    print(' | '.join(str(c) for c in fila if c))
        print()

Inspeccionando: /content/pdf/histologia_completo.pdf
Total de paginas: 58
--- Pagina 1 ---
Servicios: Corriente eléctrica.
Procedimiento.
El alumno observará las preparaciones histológicas en el microscopio con la supervisión del
profesor, identificando lo siguiente:
a) Pericondrio: capa fibrosa (fibroblastos, fibras de colágena), capa condrógena
(células condrógenas, condroblastos).
b) Cartílago elástico: fibras elásticas.
 Nidos (nichos o grupos isógenos), condrocitos, condroblastos.
  [Tablas: 1]
Laminilla No:
Tráquea 44
E. S. | Tejido:
Conectivo
Especializado | Variedad:
Cartilaginoso:
Hialino | Estructura señalada:
Pericondrio: capa fibrosa (fibroblastos,
fibras de colágena dispuestas
regularmente) y capa
condrógena
(condroblastos y células condrógenas)

--- Pagina 2 ---
Imagen 11.2. Tejido Conectivo Especializado, Cartílago Hialino Hialino.
Laminilla No: Tejido: Variedad: Estructura señalada:
Laringe 43 Conectivo Cartilaginoso: Nidos, condrocitos, matriz territorial e
Especializ

## Celda 5 - Indexar 3 colecciones en Qdrant

In [34]:
import time
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_KEY)

KEYWORDS_HISTO = [
    'conectivo', 'muscular', 'nervioso', 'hialino', 'elastico',
    'fibroso', 'oseo', 'condrocito', 'osteona', 'neurona',
    'pericondrio', 'laminilla', 'variedad', 'tejido'
]

def crear_coleccion(nombre: str, recrear: bool = False):
    try:
        qdrant.get_collection(nombre)
        if recrear:
            qdrant.delete_collection(nombre)
            raise Exception('recreando')
        print(f'  "{nombre}" ya existe - omitido (recrear=True para forzar)')
        return False
    except Exception as e:
        if 'omitido' in str(e):
            return False
        qdrant.create_collection(
            collection_name=nombre,
            vectors_config=VectorParams(size=512, distance=Distance.COSINE)
        )
        print(f'  Coleccion "{nombre}" creada')
        return True

def extraer_meta_tablas(tablas: list) -> str:
    partes = []
    for tabla in (tablas or []):
        for fila in tabla:
            fila_str = ' '.join(str(c) for c in fila if c).strip()
            if any(k in fila_str.lower() for k in KEYWORDS_HISTO):
                partes.append(fila_str)
    return ' | '.join(partes)

def indexar_coleccion(pdf_path: str, collection_name: str, modo):
    points = []
    gid = 0
    with pdfplumber.open(pdf_path) as pdf:
        for num_pag, page in enumerate(pdf.pages, 1):
            texto_pagina = page.extract_text() or ''
            meta = extraer_meta_tablas(page.extract_tables())
            texto_base = f'Pagina {num_pag}'
            if meta:
                texto_base += f' | {meta}'
            if texto_pagina:
                texto_base += f' | {texto_pagina}'
            texto_base = texto_base.strip()
            if not texto_base:
                continue
            if modo == 'pagina':
                chunks = [texto_base]
            else:
                step = max(1, int(modo * 0.8))
                chunks = [
                    texto_base[i:i+modo]
                    for i in range(0, len(texto_base), step)
                    if texto_base[i:i+modo].strip()
                ]
            for j, chunk in enumerate(chunks):
                emb = embed_texto(chunk)
                points.append(PointStruct(
                    id=gid, vector=emb,
                    payload={
                        'texto':   chunk,
                        'fuente':  os.path.basename(pdf_path),
                        'pagina':  num_pag,
                        'chunk_j': j,
                        'modo':    str(modo),
                    }
                ))
                gid += 1
    for i in range(0, len(points), 100):
        qdrant.upsert(collection_name=collection_name, points=points[i:i+100])
    print(f'  {len(points)} puntos indexados en "{collection_name}"')
    return len(points)

CONFIGS = {
    'histologia_eval_pagina': 'pagina',
    'histologia_eval_500':    500,
    'histologia_eval_250':    250,
}

for nombre_col, modo in CONFIGS.items():
    print(f'Procesando: {nombre_col} (modo={modo})')
    creada = crear_coleccion(nombre_col, recrear=True)
    if creada:
        indexar_coleccion(PDF_PATH, nombre_col, modo)

print('Indexacion completa')

Procesando: histologia_eval_pagina (modo=pagina)
  Coleccion "histologia_eval_pagina" creada
  58 puntos indexados en "histologia_eval_pagina"
Procesando: histologia_eval_500 (modo=500)
  Coleccion "histologia_eval_500" creada
  120 puntos indexados en "histologia_eval_500"
Procesando: histologia_eval_250 (modo=250)
  Coleccion "histologia_eval_250" creada
  216 puntos indexados en "histologia_eval_250"
Indexacion completa


## Celda 6 - Golden Set
Preguntas y ground truths extraidos del manual.

Verificar que `paginas_esperadas` coincida con lo visto en la Celda 4.

30 preguntas: 20 de arch2, 4 de arch3, 6 de arch4.

In [35]:
GOLDEN_SET = [
    # ── arch2 ────────────────────────────────────────────────
    {
        'question': 'Cuales son las dos capas del pericondrio y que celulas contiene cada una?',
        'ground_truth': 'El pericondrio tiene una capa fibrosa con fibroblastos y fibras de colagena, y una capa condrogena con celulas condrogenas y condroblastos.',
        'paginas_esperadas': [1]
    },
    {
        'question': 'Que estructuras se identifican en el cartilago hialino?',
        'ground_truth': 'En el cartilago hialino se identifican nidos o grupos isogenos, condrocitos, condroblastos, y matriz territorial e interterritorial.',
        'paginas_esperadas': [1, 2]
    },
    {
        'question': 'Que laminilla corresponde al cartilago hialino de traquea?',
        'ground_truth': 'La laminilla numero 44 Traquea 44 ES corresponde al cartilago hialino, donde se senala el pericondrio con su capa fibrosa y condrogena.',
        'paginas_esperadas': [1]
    },
    {
        'question': 'Cuales son las estructuras senaladas en la lamina de laringe 43?',
        'ground_truth': 'En la lamina de laringe 43 se senalan nidos, condrocitos, matriz territorial e interterritorial del cartilago hialino.',
        'paginas_esperadas': [2]
    },
    {
        'question': 'Que caracteristica distintiva tiene el cartilago elastico respecto al hialino?',
        'ground_truth': 'El cartilago elastico contiene fibras elasticas en su matriz, ademas de nidos, condrocitos, condroblastos y matriz territorial e interterritorial.',
        'paginas_esperadas': [3]
    },
    {
        'question': 'En que laminilla se observa el cartilago elastico y de que organo proviene?',
        'ground_truth': 'El cartilago elastico se observa en la laminilla de piel de oreja de raton 85a.',
        'paginas_esperadas': [3]
    },
    {
        'question': 'Que se observa en el cartilago hialino del disco de crecimiento?',
        'ground_truth': 'En el cartilago hialino del disco de crecimiento se observan hileras de condrocitos. La laminilla corresponde a hueso fracturado 79c.',
        'paginas_esperadas': [4]
    },
    {
        'question': 'Como se distingue el cartilago fibroso histologicamente?',
        'ground_truth': 'El cartilago fibroso se distingue por hileras de condrocitos y fibras de colagena dispuestas regularmente.',
        'paginas_esperadas': [5]
    },
    {
        'question': 'Cuales son los tipos de celulas del hueso compacto?',
        'ground_truth': 'Las celulas del hueso compacto son osteoprogenitoras, osteoblastos, osteocitos y osteoclastos.',
        'paginas_esperadas': [7, 8, 9]
    },
    {
        'question': 'Que es la osteona y que estructuras la componen?',
        'ground_truth': 'La osteona es la unidad estructural del hueso compacto, formada por laminas concentricas alrededor del conducto de Havers.',
        'paginas_esperadas': [10]
    },
    {
        'question': 'Que diferencia hay entre el conducto de Havers y el de Volkmann?',
        'ground_truth': 'El conducto de Havers corre longitudinalmente en el centro de la osteona. El conducto de Volkmann corre transversalmente conectando osteonas y el periostio.',
        'paginas_esperadas': [16]
    },
    {
        'question': 'Que funcion tienen los osteoclastos y donde se ubican?',
        'ground_truth': 'Los osteoclastos son celulas gigantes multinucleadas de 20 a 100 micras cuya funcion es la reabsorcion osea. Se encuentran en las lagunas de Howship.',
        'paginas_esperadas': [9]
    },
    {
        'question': 'Cuales son los tres tipos de tejido muscular y sus caracteristicas principales?',
        'ground_truth': 'Estriado voluntario con celulas cilindricas y nucleos perifericos, estriado involuntario o cardiaco con discos intercalares, y liso involuntario con celulas fusiformes y nucleo central.',
        'paginas_esperadas': [17]
    },
    {
        'question': 'Que estructuras componen la sarcomera?',
        'ground_truth': 'La sarcomera se extiende entre dos lineas Z e incluye banda A oscura, banda I clara, banda H dentro de la A, y linea M al centro de la H.',
        'paginas_esperadas': [17, 22]
    },
    {
        'question': 'Que laminilla se usa para observar el tejido muscular liso involuntario?',
        'ground_truth': 'Se usa la laminilla de estomago HE para observar los miocitos lisos involuntarios.',
        'paginas_esperadas': [21]
    },
    {
        'question': 'Como se clasifican las neuronas segun el numero de prolongaciones?',
        'ground_truth': 'Las neuronas se clasifican en unipolares, bipolares, pseudounipolares y multipolares.',
        'paginas_esperadas': [24]
    },
    {
        'question': 'En que laminilla se observan las neuronas piriformes y en grano?',
        'ground_truth': 'Las neuronas piriformes y en grano se observan en la laminilla de corteza cerebelosa 61 HE.',
        'paginas_esperadas': [27]
    },
    {
        'question': 'Cual es la diferencia entre astrocito protoplasmático y fibroso?',
        'ground_truth': 'El astrocito protoplasmático tiene prolongaciones cortas y gruesas en sustancia gris. El fibroso tiene prolongaciones largas y delgadas en sustancia blanca.',
        'paginas_esperadas': [31]
    },
    {
        'question': 'Que funcion tienen los oligodendrocitos?',
        'ground_truth': 'Los oligodendrocitos participan en la formacion de la mielina para la proteccion de los axones en el sistema nervioso central.',
        'paginas_esperadas': [31, 33]
    },
    {
        'question': 'Que son las celulas ependimarias y donde se ubican?',
        'ground_truth': 'Las celulas ependimarias revisten el canal ependimario y el conducto central de la medula espinal. Son cubicas o cilindricas.',
        'paginas_esperadas': [35]
    },
    # ── arch3 ────────────────────────────────────────────────
    {
        'question': 'Cuales son las tres tunicas de la arteria muscular y que tejido forma cada una?',
        'ground_truth': 'La arteria muscular tiene tunica intima con endotelio (epitelio plano simple) y lamina elastica interna, tunica media con musculo liso dispuesto en capas circunferenciales, y tunica adventicia con tejido conectivo colageno denso no modelado con fibroblastos y fibrocitos.',
        'paginas_esperadas': [36, 37]
    },
    {
        'question': 'Como se identifican los nucleos del endotelio en la arteria muscular?',
        'ground_truth': 'Los nucleos del endotelio se tinen de color violeta oscuro por su cromatina densa, algunos se ven alargados y otros como puntos. Forman el epitelio plano simple en contacto con la luz arterial.',
        'paginas_esperadas': [37]
    },
    {
        'question': 'Que caracteristica distingue la lamina elastica interna en un corte histologico?',
        'ground_truth': 'La lamina elastica interna se observa como una linea ondulada que se tine de color rosa translucido, ubicada entre el endotelio y la tunica media de musculo liso.',
        'paginas_esperadas': [37]
    },
    {
        'question': 'Como se diferencia una vena de una arteria muscular en corte transversal?',
        'ground_truth': 'La arteria muscular tiene pared mas gruesa con tunica media de musculo liso bien desarrollada y luz mas irregular. La vena tiene pared mas delgada y luz mas amplia e irregular.',
        'paginas_esperadas': [41, 43]
    },
    # ── arch4 ────────────────────────────────────────────────
    {
        'question': 'Cuales son los componentes del epitelio seminifero del testiculo?',
        'ground_truth': 'El epitelio seminifero esta formado por celulas germinales en distintos estadios (espermatogonias, espermatocitos, espermatides, espermatozoides) y celulas de Sertoli o sustentaculares que se extienden desde la lamina basal hasta la luz tubular.',
        'paginas_esperadas': [44, 45]
    },
    {
        'question': 'Como se distinguen las espermatogonias A claras de las A oscuras?',
        'ground_truth': 'Las espermatogonias A claras tienen nucleo redondeado con cromatina finamente granulada y nucleolos unidos a la carioteca. Las A oscuras se distinguen por una zona de rarefaccion de la cromatina en el centro del nucleo.',
        'paginas_esperadas': [44]
    },
    {
        'question': 'Cuales son las caracteristicas morfologicas de las celulas de Sertoli?',
        'ground_truth': 'Las celulas de Sertoli son celulas epiteliales columnares con gran nucleo central con pliegues en la carioteca, cromatina laxa y nucleolo evidente. Se extienden desde la lamina basal hasta la luz tubular.',
        'paginas_esperadas': [44, 45]
    },
    {
        'question': 'Donde se ubican las celulas de Leydig y cual es su funcion?',
        'ground_truth': 'Las celulas de Leydig o intersticiales se ubican en el intersticio testicular entre los tubulos seminiferos, en un estroma de tejido conectivo laxo. Son celulas polidricas grandes con citoplasma acidofilo e inclusiones lipidicas, responsables de la sintesis de androgenos.',
        'paginas_esperadas': [44, 45]
    },
    {
        'question': 'Que son las celulas peritubulares y donde se localizan?',
        'ground_truth': 'Las celulas peritubulares o mioides son celulas alargadas con caracteristicas de miofibroblastos que junto con la lamina basal del epitelio y fibras colagenas constituyen la pared tubular de los tubulos seminiferos.',
        'paginas_esperadas': [44, 45]
    },
    {
        'question': 'En que se diferencia una espermatide temprana de una tardia?',
        'ground_truth': 'Las espermatides tempranas son celulas redondas con nucleo esferico y cromatina granular palida. Las espermatides tardias tienen nucleo ahusado con cromatina muy condensada y atraviesan transformaciones celulares para dar lugar a los espermatozoides.',
        'paginas_esperadas': [44, 45]
    }
]

print(f'Golden set cargado: {len(GOLDEN_SET)} pares pregunta-respuesta')

Golden set cargado: 30 pares pregunta-respuesta


## Celda 7 - Funciones de retrieval y evaluacion

In [41]:
from langchain_groq import ChatGroq
from datasets import Dataset
from ragas import evaluate
from ragas.metrics.collections import ContextPrecision, ContextRecall, Faithfulness
from ragas.run_config import RunConfig

llm_evaluador = ChatGroq(
    model='llama-3.1-8b-instant',  # antes: llama-3.3-70b-versatile
    # 8B evalúa con menos precisión, pero para comparar configuraciones entre sí (relativo) sigue siendo válido.
    temperature=0,
    max_retries=3,
)
print('LLM evaluador listo: Groq Llama 3.1 8B')

def recuperar_contexto(pregunta: str, collection_name: str, top_k: int = 3) -> list:
    query_vec = embed_texto(pregunta)
    try:
        resultados = qdrant.query_points(
            collection_name=collection_name,
            query=query_vec,
            limit=top_k,
            with_payload=True
        )
        return [p.payload.get('texto', '') for p in resultados.points]
    except Exception as e:
        print(f'Error en retrieval ({collection_name}): {e}')
        return []

def calcular_recall_k(golden_set: list, collection_name: str, k_values=(1,3,5)) -> dict:
    recalls = {k: [] for k in k_values}
    for caso in golden_set:
        query_vec = embed_texto(caso['question'])
        try:
            resultados = qdrant.query_points(
                collection_name=collection_name,
                query=query_vec,
                limit=max(k_values),
                with_payload=True
            )
            paginas_rec = [p.payload.get('pagina') for p in resultados.points]
        except:
            paginas_rec = []
        esperadas = set(caso['paginas_esperadas'])
        for k in k_values:
            top_k_pags = set(paginas_rec[:k])
            recalls[k].append(bool(top_k_pags & esperadas))
    return {f'Recall@{k}': round(sum(v)/len(v), 3) for k, v in recalls.items()}

def evaluar_con_ragas(golden_set: list, collection_name: str) -> dict:
    filas = []
    for caso in golden_set:
        contextos = recuperar_contexto(caso['question'], collection_name, top_k=3)
        answer = contextos[0] if contextos else 'Sin contexto recuperado'
        filas.append({
            'question':     caso['question'],
            'answer':       answer,
            'contexts':     contextos,
            'ground_truth': caso['ground_truth'],
        })
        time.sleep(0.5)
    ds = Dataset.from_list(filas)
    run_config = RunConfig(max_workers=1, timeout=120, max_retries=3)

    metricas = [
        ContextPrecision(llm=llm_evaluador),
        ContextRecall(llm=llm_evaluador),
        Faithfulness(llm=llm_evaluador),
    ]

    return evaluate(
        dataset=ds,
        metrics=metricas,
        run_config=run_config,
    )

print('Funciones de evaluacion listas')

LLM evaluador listo: Groq Llama 3.1 8B
Funciones de evaluacion listas


## Celda 8 - Recall@K (rapido, sin LLM)
Correr esto primero. No consume cuota de Groq.

In [42]:
print('=' * 60)
print('EVALUACION RECALL@K (sin LLM)')
print('=' * 60)

resultados_recall = {}

for nombre_col in CONFIGS.keys():
    print(f'Evaluando: {nombre_col}...')
    recall = calcular_recall_k(GOLDEN_SET, nombre_col)
    resultados_recall[nombre_col] = recall
    for k, v in recall.items():
        print(f'   {k}: {v:.3f}')

print()
print(f'{"Configuracion":<35} {"Recall@1":>10} {"Recall@3":>10} {"Recall@5":>10}')
print('-' * 70)
for nombre, r in resultados_recall.items():
    print(f'{nombre:<35} {r["Recall@1"]:>10.3f} {r["Recall@3"]:>10.3f} {r["Recall@5"]:>10.3f}')

mejor = max(resultados_recall, key=lambda x: resultados_recall[x]['Recall@3'])
print(f'Mejor por Recall@3: {mejor}')

EVALUACION RECALL@K (sin LLM)
Evaluando: histologia_eval_pagina...
   Recall@1: 0.167
   Recall@3: 0.300
   Recall@5: 0.333
Evaluando: histologia_eval_500...
   Recall@1: 0.167
   Recall@3: 0.200
   Recall@5: 0.367
Evaluando: histologia_eval_250...
   Recall@1: 0.133
   Recall@3: 0.233
   Recall@5: 0.267

Configuracion                         Recall@1   Recall@3   Recall@5
----------------------------------------------------------------------
histologia_eval_pagina                   0.167      0.300      0.333
histologia_eval_500                      0.167      0.200      0.367
histologia_eval_250                      0.133      0.233      0.267
Mejor por Recall@3: histologia_eval_pagina


In [43]:
for nombre_col in CONFIGS.keys():
    print(f'\n=== {nombre_col} ===')
    for seccion, indices in [('arch2', range(0,20)), ('arch3', range(20,24)), ('arch4', range(24,30))]:
        subset = [GOLDEN_SET[i] for i in indices]
        r = calcular_recall_k(subset, nombre_col)
        print(f'  {seccion}: R@1={r["Recall@1"]:.3f} R@3={r["Recall@3"]:.3f} R@5={r["Recall@5"]:.3f}')


=== histologia_eval_pagina ===
  arch2: R@1=0.100 R@3=0.150 R@5=0.200
  arch3: R@1=0.000 R@3=0.250 R@5=0.250
  arch4: R@1=0.500 R@3=0.833 R@5=0.833

=== histologia_eval_500 ===
  arch2: R@1=0.200 R@3=0.200 R@5=0.400
  arch3: R@1=0.000 R@3=0.000 R@5=0.000
  arch4: R@1=0.167 R@3=0.333 R@5=0.500

=== histologia_eval_250 ===
  arch2: R@1=0.200 R@3=0.300 R@5=0.350
  arch3: R@1=0.000 R@3=0.000 R@5=0.000
  arch4: R@1=0.000 R@3=0.167 R@5=0.167


## Celda 9 - Metricas RAGAS completas (usa Groq)
Correr solo si el Recall@K muestra diferencias claras entre configuraciones.
Consume aprox 90 llamadas al LLM en total.

BLOQUEANTE: RAGAS >= 0.2 requiere InstructorLLM (OpenAI).
ChatGroq no es compatible con las métricas de colecciones en esta versión.
Pendiente para Sprint futuro con evaluador OpenAI o downgrade a ragas==0.1.21.

In [44]:
print('=' * 60)
print('EVALUACION RAGAS COMPLETA')
print('=' * 60)

resultados_ragas = {}

for nombre_col in CONFIGS.keys():
    print(f'Evaluando: {nombre_col}...')
    try:
        resultado = evaluar_con_ragas(GOLDEN_SET, nombre_col)
        resultados_ragas[nombre_col] = resultado
        print(f'   Context Precision : {resultado["context_precision"]:.3f}')
        print(f'   Context Recall    : {resultado["context_recall"]:.3f}')
        print(f'   Faithfulness      : {resultado["faithfulness"]:.3f}')
    except Exception as e:
        print(f'   Error: {e}')
        resultados_ragas[nombre_col] = None
    print('   Esperando 15s...')
    time.sleep(15)

print('Evaluacion RAGAS completa')

EVALUACION RAGAS COMPLETA
Evaluando: histologia_eval_pagina...
   Error: Collections metrics only support modern InstructorLLM. Found: ChatGroq. Use: llm_factory('gpt-4o-mini', client=openai_client)
   Esperando 15s...
Evaluando: histologia_eval_500...
   Error: Collections metrics only support modern InstructorLLM. Found: ChatGroq. Use: llm_factory('gpt-4o-mini', client=openai_client)
   Esperando 15s...
Evaluando: histologia_eval_250...
   Error: Collections metrics only support modern InstructorLLM. Found: ChatGroq. Use: llm_factory('gpt-4o-mini', client=openai_client)
   Esperando 15s...
Evaluacion RAGAS completa


## Celda 10 - Tabla comparativa final

In [46]:
import json

print('=' * 75)
print('TABLA COMPARATIVA FINAL')
print('=' * 75)

header = f'{"Configuracion":<35} {"R@1":>5} {"R@3":>5} {"R@5":>5} {"Prec":>6} {"Rec":>6} {"Faith":>6}'
print(header)
print('-' * 75)

for nombre in CONFIGS.keys():
    rk = resultados_recall.get(nombre, {})
    rr = resultados_ragas.get(nombre)
    r1    = f'{rk.get("Recall@1", 0):.2f}'
    r3    = f'{rk.get("Recall@3", 0):.2f}'
    r5    = f'{rk.get("Recall@5", 0):.2f}'
    prec  = f'{rr["context_precision"]:.2f}' if rr else ' N/A'
    rec   = f'{rr["context_recall"]:.2f}'    if rr else ' N/A'
    faith = f'{rr["faithfulness"]:.2f}'      if rr else ' N/A'
    print(f'{nombre:<35} {r1:>5} {r3:>5} {r5:>5} {prec:>6} {rec:>6} {faith:>6}')

resultados_export = {
    'recall_k': resultados_recall,
    'ragas': {
        k: {
            'context_precision': v['context_precision'] if v else None,
            'context_recall':    v['context_recall']    if v else None,
            'faithfulness':      v['faithfulness']      if v else None,
        }
        for k, v in resultados_ragas.items()
    }
}

with open('/content/resultados_chunk_eval.json', 'w') as f:
    json.dump(resultados_export, f, indent=2, ensure_ascii=False)
print('Resultados guardados en /content/resultados_chunk_eval.json')

if resultados_recall:
    mejor_r3 = max(resultados_recall, key=lambda x: resultados_recall[x]['Recall@3'])
    print(f'Recomendacion: usar "{mejor_r3}"')
    print(f'Recall@3 = {resultados_recall[mejor_r3]["Recall@3"]:.3f}')
    print()
    print('Interpretacion:')
    print('  Recall@3 > 0.75  configuracion lista para produccion')
    print('  Recall@3 0.5-0.75  revisar chunking o threshold')
    print('  Recall@3 < 0.5   problema estructural en ingestion')

TABLA COMPARATIVA FINAL
Configuracion                         R@1   R@3   R@5   Prec    Rec  Faith
---------------------------------------------------------------------------
histologia_eval_pagina               0.17  0.30  0.33    N/A    N/A    N/A
histologia_eval_500                  0.17  0.20  0.37    N/A    N/A    N/A
histologia_eval_250                  0.13  0.23  0.27    N/A    N/A    N/A
Resultados guardados en /content/resultados_chunk_eval.json
Recomendacion: usar "histologia_eval_pagina"
Recall@3 = 0.300

Interpretacion:
  Recall@3 > 0.75  configuracion lista para produccion
  Recall@3 0.5-0.75  revisar chunking o threshold
  Recall@3 < 0.5   problema estructural en ingestion


---
## Diagnostico: imagen vs imagen

Las siguientes celdas diagnostican si el sistema recupera correctamente
la pagina esperada cuando se le pasa una imagen como query.

**Orden:**
1. Celda 11 — instalar CONCH
2. Celda 12 — test imagen->imagen con CLIP
3. Celda 13 — test imagen->imagen con CONCH
4. Celda 14 — reporte comparativo CLIP vs CONCH

> Prerrequisito: haber corrido las celdas 1-5 (CLIP cargado, colecciones indexadas).

## Celda 11 — Instalar CONCH
> Requiere `HUGGINGFACE_TOKEN` como secreto en Colab.
> Aceptar terminos en huggingface.co/MahmoodLab/conch antes de correr.

In [47]:
# CONCH requiere aceptar los terminos en HuggingFace antes de descargar.
# Ir a: https://huggingface.co/MahmoodLab/conch y aceptar el acuerdo.
# Luego agregar HUGGINGFACE_TOKEN como secreto en Colab.

!pip install --quiet timm huggingface_hub

from google.colab import userdata as _ud
HF_TOKEN = _ud.get('HF_TOKEN')

if HF_TOKEN:
    import os
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HuggingFace token cargado')
else:
    print('ADVERTENCIA: HF_TOKEN no encontrado.')
    print('CONCH no podra descargarse. Solo se ejecutara el test con CLIP.')
    HF_TOKEN = None

HuggingFace token cargado


## Celda 12 - Test imagen->imagen con CLIP

In [48]:
from pdf2image import convert_from_path
import torch, numpy as np

def test_imagen_imagen(pdf_path, collection_name, modelo_nombre='CLIP',
                        embed_fn=None, top_k=5):
    """
    Por cada pagina del PDF genera su embedding y busca en Qdrant.
    Mide si la pagina correcta aparece en los top-K resultados.
    embed_fn: funcion que recibe PIL Image y devuelve list[float]
    """
    if embed_fn is None:
        def embed_fn(pil_img):
            inputs = clip_processor(images=pil_img, return_tensors='pt').to(device)
            with torch.no_grad():
                feats = clip_model.get_image_features(**inputs)
                if hasattr(feats, 'pooler_output'):
                    emb = feats.pooler_output.cpu().numpy().flatten()
                else:
                    emb = feats.cpu().numpy().flatten()
            return (emb / np.linalg.norm(emb)).tolist()

    print(f'Renderizando paginas de {pdf_path}...')
    paginas = convert_from_path(pdf_path, dpi=150)
    print(f'Total paginas: {len(paginas)}')

    resultados = []
    for i, pil_img in enumerate(paginas, 1):
        emb = embed_fn(pil_img)
        try:
            hits = qdrant.query_points(
                collection_name=collection_name,
                query=emb,
                limit=top_k,
                with_payload=True
            )
            paginas_rec = [h.payload.get('pagina') for h in hits.points]
            scores_rec  = [round(h.score, 4) for h in hits.points]
            score_top1  = scores_rec[0] if scores_rec else 0
        except Exception as e:
            paginas_rec = []
            scores_rec  = []
            score_top1  = 0
            print(f'  Error pagina {i}: {e}')

        hit_top1 = (paginas_rec[0] == i) if paginas_rec else False
        hit_top3 = i in paginas_rec[:3]
        hit_top5 = i in paginas_rec[:5]

        resultados.append({
            'pagina':      i,
            'top1_pagina': paginas_rec[0] if paginas_rec else None,
            'top1_score':  score_top1,
            'hit_top1':    hit_top1,
            'hit_top3':    hit_top3,
            'hit_top5':    hit_top5,
            'paginas_rec': paginas_rec,
            'scores_rec':  scores_rec,
        })

    n = len(resultados)
    r1 = sum(r['hit_top1'] for r in resultados) / n
    r3 = sum(r['hit_top3'] for r in resultados) / n
    r5 = sum(r['hit_top5'] for r in resultados) / n

    print(f'\n=== {modelo_nombre} | {collection_name} ===')
    print(f'  Recall@1: {r1:.3f}  ({int(r1*n)}/{n})')
    print(f'  Recall@3: {r3:.3f}  ({int(r3*n)}/{n})')
    print(f'  Recall@5: {r5:.3f}  ({int(r5*n)}/{n})')

    # Diagnostico de fallas
    fallas = [r for r in resultados if not r['hit_top3']]
    if fallas:
        print(f'\n  Paginas que fallan en top-3 ({len(fallas)}):')
        for r in fallas:
            causa = ''
            if r['top1_score'] < 0.22:
                causa = f'score bajo ({r["top1_score"]}), posible threshold o imagen pequena'
            elif r['top1_score'] >= 0.22:
                causa = f'score ok ({r["top1_score"]}) pero pagina incorrecta, problema de chunk'
            print(f'    Pag {r["pagina"]:3} -> recupero pag {r["top1_pagina"]} | {causa}')
    else:
        print('  Sin fallas en top-3')

    return resultados, {'R@1': r1, 'R@3': r3, 'R@5': r5}


# Correr con la coleccion de pagina completa (la mas relevante para imagen->imagen)
resultados_clip, metricas_clip = test_imagen_imagen(
    pdf_path=PDF_PATH,
    collection_name='histologia_eval_pagina',
    modelo_nombre='CLIP'
)

Renderizando paginas de /content/pdf/histologia_completo.pdf...
Total paginas: 58

=== CLIP | histologia_eval_pagina ===
  Recall@1: 0.000  (0/58)
  Recall@3: 0.034  (2/58)
  Recall@5: 0.069  (4/58)

  Paginas que fallan en top-3 (56):
    Pag   1 -> recupero pag 33 | score ok (0.3385) pero pagina incorrecta, problema de chunk
    Pag   2 -> recupero pag 23 | score ok (0.2875) pero pagina incorrecta, problema de chunk
    Pag   3 -> recupero pag 23 | score ok (0.3225) pero pagina incorrecta, problema de chunk
    Pag   4 -> recupero pag 23 | score ok (0.2973) pero pagina incorrecta, problema de chunk
    Pag   5 -> recupero pag 17 | score ok (0.2941) pero pagina incorrecta, problema de chunk
    Pag   6 -> recupero pag 23 | score ok (0.3182) pero pagina incorrecta, problema de chunk
    Pag   7 -> recupero pag 10 | score ok (0.3135) pero pagina incorrecta, problema de chunk
    Pag   8 -> recupero pag 10 | score ok (0.294) pero pagina incorrecta, problema de chunk
    Pag   9 -> recupe

## Celda 13 - Test imagen->imagen con CONCH

In [49]:
# Solo corre si HF_TOKEN esta disponible
resultados_conch = None
metricas_conch   = None

if not HF_TOKEN:
    print('CONCH omitido: no hay HF_TOKEN configurado.')
else:
    try:
        import os
        # Eliminar token viejo de Colab para evitar conflictos
        if 'HF_TOKEN' in os.environ:
            del os.environ['HF_TOKEN']

        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        os.environ['HF_TOKEN'] = HF_TOKEN
        os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
        print('Login HuggingFace OK')

        # ── Paso 1: Descargar checkpoint ──────────────────────────────
        print('Descargando checkpoint CONCH...')
        from huggingface_hub import hf_hub_download
        import torch

        checkpoint_path = hf_hub_download(
            repo_id='MahmoodLab/conch',
            filename='pytorch_model.bin',
            token=HF_TOKEN
        )
        print(f'  Checkpoint: {checkpoint_path}')
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        print(f'  Keys totales: {len(checkpoint)}')

        # ── Paso 2: Cargar encoder visual con pesos de CONCH ─────────
        print('Cargando encoder visual CONCH...')
        import timm

        # timm usa la misma nomenclatura que CONCH (trunk.)
        conch_vit = timm.create_model(
            'vit_base_patch16_224',
            pretrained=False,
            num_classes=0  # sin cabeza de clasificacion, devuelve features
        )

        # Extraer pesos visuales sin el prefijo 'visual.'
        visual_sd = {
            k.replace('visual.', ''): v
            for k, v in checkpoint.items()
            if k.startswith('visual.')
        }

        # Quitar proj_contrast que no pertenece al ViT
        visual_sd.pop('proj_contrast', None)

        missing, unexpected = conch_vit.load_state_dict(visual_sd, strict=False)
        print(f'  Missing keys : {len(missing)}')
        print(f'  Unexpected   : {len(unexpected)}')

        conch_vit = conch_vit.to(device)
        conch_vit.eval()

        # Transformacion estandar ImageNet (igual que antes)
        from torchvision import transforms
        conch_transform = transforms.Compose([
            transforms.Resize(224),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225]),
        ])

        # ── Paso 3: Definir funcion de embedding ──────────────────────
        def embed_conch(pil_img):
            tensor = conch_transform(pil_img).unsqueeze(0).to(device)
            with torch.no_grad():
                emb = conch_vit(tensor)
                if isinstance(emb, (list, tuple)):
                    emb = emb[0]
                emb = emb.cpu().numpy().flatten()
            return (emb / np.linalg.norm(emb)).tolist()

        # Test
        from PIL import Image
        test_emb = embed_conch(Image.new('RGB', (224, 224)))
        print(f'  Test embedding OK | dimension: {len(test_emb)}')
        print('CONCH listo')

        # ── Paso 4: Indexar en Qdrant con embeddings CONCH ───────────
        print('Indexando con CONCH...')
        crear_coleccion('histologia_conch_pagina', recrear=True)

        from pdf2image import convert_from_path
        import pdfplumber
        from qdrant_client.models import PointStruct, VectorParams, Distance

        paginas = convert_from_path(PDF_PATH, dpi=150)
        points_conch = []

        with pdfplumber.open(PDF_PATH) as pdf:
            for i, (pil_img, page) in enumerate(zip(paginas, pdf.pages), 1):
                texto = page.extract_text() or ''
                meta  = extraer_meta_tablas(page.extract_tables())
                texto_base = f'Pagina {i}'
                if meta:  texto_base += f' | {meta}'
                if texto: texto_base += f' | {texto}'

                emb = embed_conch(pil_img)

                # En la primera pagina, ajustar dimension de la coleccion
                if i == 1:
                    dim_conch = len(emb)
                    print(f'  Dimension CONCH: {dim_conch}')
                    qdrant.delete_collection('histologia_conch_pagina')
                    qdrant.create_collection(
                        collection_name='histologia_conch_pagina',
                        vectors_config=VectorParams(size=dim_conch, distance=Distance.COSINE)
                    )

                points_conch.append(PointStruct(
                    id=i-1, vector=emb,
                    payload={
                        'texto':  texto_base[:600],
                        'fuente': os.path.basename(PDF_PATH),
                        'pagina': i,
                    }
                ))

        qdrant.upsert(collection_name='histologia_conch_pagina', points=points_conch)
        print(f'  {len(points_conch)} paginas indexadas con CONCH')

        # ── Paso 5: Test imagen->imagen con CONCH ─────────────────────
        resultados_conch, metricas_conch = test_imagen_imagen(
            pdf_path=PDF_PATH,
            collection_name='histologia_conch_pagina',
            modelo_nombre='CONCH',
            embed_fn=embed_conch
        )

    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f'Error: {e}')

Login HuggingFace OK
Descargando checkpoint CONCH...
  Checkpoint: /root/.cache/huggingface/hub/models--MahmoodLab--conch/snapshots/f9ca9f877171a28ade80228fb195ac5d79003357/pytorch_model.bin
  Keys totales: 326
Cargando encoder visual CONCH...
  Missing keys : 150
  Unexpected   : 174
  Test embedding OK | dimension: 768
CONCH listo
Indexando con CONCH...
  Coleccion "histologia_conch_pagina" creada
  Dimension CONCH: 768
  58 paginas indexadas con CONCH
Renderizando paginas de /content/pdf/histologia_completo.pdf...
Total paginas: 58

=== CONCH | histologia_conch_pagina ===
  Recall@1: 1.000  (58/58)
  Recall@3: 1.000  (58/58)
  Recall@5: 1.000  (58/58)
  Sin fallas en top-3


## Celda 14 - Reporte comparativo CLIP vs CONCH

In [50]:
import json

print('=' * 65)
print('REPORTE DIAGNOSTICO IMAGEN->IMAGEN')
print('=' * 65)

# Tabla de metricas
print(f'\n{"Modelo":<12} {"R@1":>6} {"R@3":>6} {"R@5":>6}')
print('-' * 35)
if metricas_clip:
    print(f'{"CLIP":<12} {metricas_clip["R@1"]:>6.3f} {metricas_clip["R@3"]:>6.3f} {metricas_clip["R@5"]:>6.3f}')
if metricas_conch:
    print(f'{"CONCH":<12} {metricas_conch["R@1"]:>6.3f} {metricas_conch["R@3"]:>6.3f} {metricas_conch["R@5"]:>6.3f}')

# Clasificar fallas por causa
if resultados_clip:
    fallas = [r for r in resultados_clip if not r['hit_top3']]
    bajo_threshold  = [r for r in fallas if r['top1_score'] < 0.22]
    chunk_incorrecto = [r for r in fallas if r['top1_score'] >= 0.22]

    print(f'\nDiagnostico CLIP ({len(fallas)} fallas en top-3):')
    print(f'  Score < 0.22 (threshold/imagen pequenya): {len(bajo_threshold)} paginas')
    if bajo_threshold:
        pags = [r["pagina"] for r in bajo_threshold]
        print(f'    Paginas afectadas: {pags}')
        print(f'    -> Accion: probar DPI=200 o bajar threshold')
    print(f'  Score ok pero pagina incorrecta (chunk):  {len(chunk_incorrecto)} paginas')
    if chunk_incorrecto:
        pags = [r["pagina"] for r in chunk_incorrecto]
        print(f'    Paginas afectadas: {pags}')
        print(f'    -> Accion: revisar texto enriquecido de esas paginas')

# Recomendacion final
print('\nRecomendacion:')
r3_clip  = metricas_clip['R@3']  if metricas_clip  else 0
r3_conch = metricas_conch['R@3'] if metricas_conch else 0

if r3_clip >= 0.75 and r3_conch == 0:
    print('  CLIP suficiente (Recall@3 >= 0.75). No es necesario migrar a CONCH.')
elif r3_conch > r3_clip + 0.05:
    print('  CONCH supera a CLIP en mas de 5 puntos. Recomendado migrar.')
    print('  Recordar: coordinar con otros grupos (G1, G3) para que usen el mismo')
    print('  modelo evaluador en la comparativa del Sprint 6.')
elif r3_clip < 0.5:
    print('  Recall@3 < 0.5: problema estructural.')
    print('  Verificar que las imagenes esten indexadas en la coleccion correcta.')
    print('  Probar aumentar DPI a 200 y reindexa.')
else:
    print(f'  CLIP Recall@3={r3_clip:.2f}. Aceptable pero mejorable.')
    print('  Considerar bajar SIMILARITY_THRESHOLD de 0.22 a 0.18.')

# Guardar para coordinacion con otros grupos
reporte = {
    'modelo_embedding': 'CLIP ViT-B/32 (texto) + CONCH ViT-B/16 (imagen)',
    'pdf': os.path.basename(PDF_PATH),
    'coleccion': 'histologia_eval_pagina',
    'metricas_clip':  metricas_clip,
    'metricas_conch': metricas_conch,
    'fallas_clip': [
        {'pagina': r['pagina'], 'score': r['top1_score'], 'recupero': r['top1_pagina']}
        for r in (resultados_clip or [])
        if not r['hit_top3']
    ],
}
with open('/content/reporte_diagnostico_imagenes.json', 'w') as f:
    json.dump(reporte, f, indent=2)
print('\nReporte guardado en /content/reporte_diagnostico_imagenes.json')
print('Compartir con G1, G3 y equipo de metricas para analisis cruzado.')

REPORTE DIAGNOSTICO IMAGEN->IMAGEN

Modelo          R@1    R@3    R@5
-----------------------------------
CLIP          0.000  0.034  0.069
CONCH         1.000  1.000  1.000

Diagnostico CLIP (56 fallas en top-3):
  Score < 0.22 (threshold/imagen pequenya): 0 paginas
  Score ok pero pagina incorrecta (chunk):  56 paginas
    Paginas afectadas: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58]
    -> Accion: revisar texto enriquecido de esas paginas

Recomendacion:
  CONCH supera a CLIP en mas de 5 puntos. Recomendado migrar.
  Recordar: coordinar con otros grupos (G1, G3) para que usen el mismo
  modelo evaluador en la comparativa del Sprint 6.

Reporte guardado en /content/reporte_diagnostico_imagenes.json
Compartir con G1, G3 y equipo de metricas para analisis cruzado.
